# 10: Speculative Decoding

Standard autoregressive decoding: generate one token per forward pass (m=1).
Each pass loads the full weight matrix (~1.6MB per GEMM for BitNet 2B).
Memory-bandwidth bound: 6-7 GB/s on a laptop CPU.

Speculative decoding: draft K candidate tokens cheaply, verify all K in one
batched forward pass (m=K). Weight data loaded once instead of K times.

| method | draft source | extra model? |
|:---|:---|:---|
| prompt lookup | n-gram match from prompt | no |
| assisted generation | small draft model | yes |

Both built into HuggingFace generate(). No smelt code changes needed.

In [ ]:
import os
import sys
import time

if "COLAB_GPU" in os.environ:
    os.system("pip install -q git+https://github.com/PritRaj1/smelt.git")
    os.system("pip install -q 'transformers>=4.51,<5'")
else:
    sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd())))

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

import smelt

In [ ]:
MODEL_ID = "microsoft/bitnet-b1.58-2B-4T"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.float32)
model.eval()
smelt.quantize(model)

## Baseline: greedy decoding

In [ ]:
prompt = "Explain how a CPU executes instructions step by step:"
ids = tokenizer.encode(prompt, return_tensors="pt")


def bench(label, gen_kwargs, n_tokens=32, runs=2):
    for _ in range(1):
        with torch.no_grad():
            model.generate(ids, max_new_tokens=n_tokens, do_sample=False, **gen_kwargs)
    times = []
    for _ in range(runs):
        t0 = time.perf_counter()
        with torch.no_grad():
            out = model.generate(ids, max_new_tokens=n_tokens, do_sample=False, **gen_kwargs)
        times.append(time.perf_counter() - t0)
    tps = n_tokens / (sum(times) / len(times))
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    print(f"{label}: {tps:.1f} tok/s")
    print(f"  {text[:120]}...")
    return tps


tps_base = bench("standard", {})

## Speculative: prompt lookup decoding

Reuse n-grams from the prompt as draft tokens. Works best when the model
repeats or paraphrases parts of the prompt (common in instruction following).

Zero extra memory, no draft model, 1 param.

In [ ]:
tps_lookup = bench("prompt lookup (k=4)", {"prompt_lookup_num_tokens": 4})

In [ ]:
import matplotlib.pyplot as plt
import torch.nn as nn

from smelt.matmul import TernaryLinear

tl = TernaryLinear(nn.Linear(2560, 2560, bias=False))
ms = []
tpts = []
for m in [1, 2, 4, 8, 16, 32]:
    x = torch.randn(m, 2560)
    tl(x)
    t0 = time.perf_counter()
    for _ in range(100):
        tl(x)
    t = (time.perf_counter() - t0) / 100 / m * 1e6
    ms.append(m)
    tpts.append(t)

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(ms, tpts, "o-")
ax.set_xlabel("batch size (m)")
ax.set_ylabel("us/token")
ax.set_title("GEMM throughput: weight loads amortized across batch")
plt.tight_layout()
plt.show()

In [ ]:
results = {"standard": tps_base, "prompt lookup (k=4)": tps_lookup}

fig, ax = plt.subplots(figsize=(6, 3))
ax.barh(list(results.keys()), list(results.values()), color=["steelblue", "coral"])
ax.set_xlabel("tok/s")
for i, v in enumerate(results.values()):
    ax.text(v + 0.2, i, f"{v:.1f}", va="center", fontsize=9)

plt.tight_layout()
plt.show()